In [ ]:
import torch
import gc

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline


ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

In [2]:
# Install required packages (run once in a Kaggle notebook)
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

# Python imports
import os
import re
import uuid
import shutil
from typing import List, Dict, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi

# LangChain imports (community modules for Kaggle)
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline

# LangChain core utilities
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain


INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 88.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━

2025-11-16 14:49:50.653442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763304591.028130      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763304591.124377      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

        # Initialize LLM for answer generation
        self.llm = self._init_llm()

    def _init_llm(self) -> Optional[HuggingFacePipeline]:
        """Initialize LLM for answer generation"""
        try:
            
            model_name = "Qwen/Qwen2.5-3B-Instruct"
            print("🤖 Loading LLM for answer generation..."+ model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                top_p=0.95
            )
            llm = HuggingFacePipeline(pipeline=pipe)
            print("✅ LLM loaded successfully")
            return llm
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Will use extractive answers only")
            return None

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         SEARCH + ANSWER
    # =======================================================================
    def search(self, query: str, k: int = 10) -> str:
        """Search for documents and generate a structured legal answer."""
        print(f"\n🔍 QUERY: {query}")

        # Detect case number in query
        case_num = self._detect_case_number(query)

        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Retrieve more results to allow filtering
                results = self._retrieve_documents(query, k=k*3)

                # Filter to exact case match (number + year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]

                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    results = results[:k]
                    print(f"⚠️ No exact case found, returning top semantic matches")
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        # Print only the filenames of retrieved PDFs
        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        # Generate answer using all retrieved documents
        answer_text = self.generate_answer(query, results)

        # Print clean structured answer
        print("\n📌 LLM Answer:\n")

        return answer_text

    # =======================================================================
    #                         ANSWER GENERATION
    # =======================================================================
    def generate_answer(self, query: str, documents: List[Document], use_llm: bool = True) -> str:
        """Generate a structured answer using all retrieved documents."""
        if not documents:
            return "No relevant documents found."

        # Build context from all documents
        context = "\n\n---\n\n".join([
            f"From {d.metadata.get('source_file', 'unknown')}:\n{d.page_content}"
            for d in documents
        ])

        if self.llm and use_llm:
            prompt = f"""

            You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using ONLY the retrieved documents provided to you.

You operate in two modes:

======================================================================
MODE 1 — STRICT CASE MODE
======================================================================
Triggered when the user query refers to a specific case number or appeal, e.g.:
"CPLA 210 of 2024", "Civil Appeal 152/2019", "C.P.L.A 47 2024", etc.

Rules:
1. You MUST identify the retrieved document(s) that correspond to that specific case, even if naming varies (e.g., CPLA / C.P.L.A / C.P.L.A. are treated as equivalent).
2. In STRICT mode, you may ONLY use the matching case document(s).
3. Ignore all other retrieved documents completely.
4. If the matching case is not present in the retrieved documents, respond:
   "The retrieved documents do not contain the required information."
5. You must never hallucinate missing details.
6. If a retrieved document is irrelevant to the query, do not use it.

======================================================================
MODE 2 — LENIENT TOPIC MODE
======================================================================
Triggered when the user asks a general or semantic question, e.g.:
"cases on domestic violence", "similar cases", "cases about nomination papers", 
"precedents about election symbols", etc.

Rules:
1. You may use ANY of the retrieved documents.
2. You must synthesize everything into ONE unified answer.
3. Never produce multiple separate answers per document.
4. Quote ONLY text that appears in the retrieved documents.
5.You must ignore irrelevant retrieved documents completely.


======================================================================
UNIVERSAL RULES (APPLY TO BOTH MODES)
======================================================================
- Use only information contained in the retrieved documents.
- Never invent or guess facts (judges, parties, citations, reasoning, legal rules).
- If the answer is not found in the retrieved documents, say so explicitly.
- You must follow the output format EXACTLY as required below.
- Do NOT include analysis, chain of thought, system messages, reasoning steps, or metadata.
- Provide a single, clean, professional legal answer.

======================================================================
MANDATORY OUTPUT FORMAT
======================================================================

Answer:
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)

======================================================================

BEGIN NOW.
            
            """

            # Append retrieved documents to prompt
            prompt += "\n\nRETRIEVED DOCUMENTS:\n" + context

            try:
                result = self.llm.invoke(prompt)
                
                # Extract the generated text
                if isinstance(result, list) and len(result) > 0:
                    answer_text = result[0].get("generated_text", str(result))
                elif isinstance(result, str):
                    answer_text = result
                else:
                    answer_text = str(result)
                
                # Remove everything before "Answer:"
                if "Answer:" in answer_text:
                    answer_text = answer_text.split("Answer:")[-1].strip()
                
                # Remove common artifact patterns (but keep the actual answer)
                # Only remove if they appear at the START of the text
                start_artifacts = ["Q:", "Question:", "RETRIEVED", "From", "Documents:"]
                for artifact in start_artifacts:
                    if answer_text.startswith(artifact):
                        answer_text = answer_text.split("\n", 1)[-1].strip()
                
                # Clean up leading/trailing whitespace
                answer_text = answer_text.strip()
                
                print(answer_text)
                # Final output with sources
                return answer_text
                
            except Exception as e:
                print(f"⚠️ LLM error: {e}")
                return self._extractive_answer(documents)
        else:
            return self._extractive_answer(documents)


In [4]:
print("=" * 70)
print("COPYING DB TO WRITABLE LOCATION")
print("=" * 70)

# Copy from read-only input to writable workspace
src = "/kaggle/input/fyp-vector-store/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)

print(f"\nCopying from: {src}")
print(f"Copying to: {dst}")
shutil.copytree(src, dst)
print("✅ Copied successfully")

# Now load from writable location
print("\n📚 Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

print("\n🔧 Initializing agent...")

COPYING DB TO WRITABLE LOCATION

Copying from: /kaggle/input/fyp-vector-store/chroma_db
Copying to: /kaggle/working/chroma_db
✅ Copied successfully

📚 Loading embeddings...


/tmp/ipykernel_48/529954725.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


🔧 Initializing agent...


In [5]:
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    embeddings=embeddings,
    db_path="/kaggle/working/chroma_db",  # ← Use writable location
    test_mode=False
)

print("\n📦 Loading vector store...")
agent.vectorstore = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embeddings
)
chunk_count = agent.vectorstore._collection.count()
print(f"✅ Loaded {chunk_count} chunks")

print("\n🔨 Rebuilding BM25...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

print("\n" + "=" * 70)
print("✅ READY TO USE")
print("=" * 70)

🤖 Loading LLM for answer generation...Qwen/Qwen2.5-3B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
/tmp/ipykernel_48/3794137242.py:55: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
/tmp/ipykernel_48/1453110868.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  agent.vectorstore = Chroma(


✅ LLM loaded successfully

📦 Loading vector store...
✅ Loaded 21895 chunks

🔨 Rebuilding BM25...
✅ BM25 ready with 21895 documents

✅ READY TO USE


In [6]:
query = "What was CPLA 210 of 2024 about?"
answer = agent.search(query, k=5)


🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
✅ Found exact case match

📄 Retrieved PDFs:
 - C.P.L.A.210_2024.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)


BEGIN NOW.
            
            

RETRIEVED DOCUMENTS:
From C.P.L.A.210_2024.pdf:
[Case Type: C.P.L.A.210] [Year: 2024] [Case Number: C.P.L.A.210_2024]

C.P.L.As.210 and 212/2024 
 
 
-:2:-
candidates for the General Elections of 2024. This candidate shall 
immediately and forthwith, and it shall be the duty of the Election 
Commission to ensure that this is done, be allocated an e lection 
symbol. (We may note that for this consti

In [9]:
query = "What was the dispute raised in Civil Petition for Leave to Appeal No. 210 of 2024 before the Supreme Court of Pakistan?"
answer = agent.search(query, k=20)


🔍 QUERY: What was the dispute raised in Civil Petition for Leave to Appeal No. 210 of 2024 before the Supreme Court of Pakistan?

📄 Retrieved PDFs:
 - C.P.L.A.181-Q_2021.pdf
 - C.P.L.A.5029_2024.pdf
 - C.P.L.A.2148-L_2022.pdf
 - C.P.L.A.3179-L_2023.pdf
 - C.P.L.A.862_2024.pdf
 - C.P.L.A.1737-L_2020.pdf
 - C.P.L.A.662-K_2024.pdf
 - C.P.L.A.520-K_2024.pdf
 - C.P.L.A.2313_2024.pdf
 - C.P.L.A.2768-L_2022.pdf
 - C.P.L.A.181_2024.pdf
 - C.M.A.12587_2021.pdf
 - C.R.P.312_2024.pdf
 - C.A.2026_2022.pdf
 - C.P.L.A.3151_2021.pdf
 - C.P.L.A.2477-L_2015.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.5718_2021.pdf
 - C.A.17-Q_2023.pdf
 - Crl.A.144-L_2020.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as man

In [11]:
query = "Why did the Supreme Court dismiss the CPLA No. 210 of 2024? What reasoning did the Court provide?"
answer = agent.search(query, k=20)


🔍 QUERY: Why did the Supreme Court dismiss the CPLA No. 210 of 2024? What reasoning did the Court provide?

📄 Retrieved PDFs:
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.251-Q_2024.pdf
 - C.P.L.A.2007-L_2023.pdf
 - C.P.L.A.3209_2020.pdf
 - C.A.316_2022.pdf
 - C.P.L.A.3755_2022.pdf
 - C.R.P.1077_2023.pdf
 - C.A.1509_2021.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.121_2024.pdf
 - C.P.L.A.1106_2024.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.148-L_2024.pdf
 - C.P.L.A.1787-L_2022.pdf
 - C.P.L.A.520-K_2024.pdf
 - C.P.L.A.2009_2025.pdf
 - C.P.L.A.3210-L_2023.pdf
 - C.P.L.A.2477_2024.pdf
 - C.P.L.A.1573_2024.pdf
 - C.A.799_2015.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only

In [12]:
query = "What findings of the High Court were challenged before the Supreme Court in CPLA No. 210 of 2024?"
answer = agent.search(query, k=20)


🔍 QUERY: What findings of the High Court were challenged before the Supreme Court in CPLA No. 210 of 2024?

📄 Retrieved PDFs:
 - C.P.L.A.1106_2024.pdf
 - C.P.L.A.251-Q_2024.pdf
 - C.P.L.A.1189_2025.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.2009_2025.pdf
 - I.C.A.1_2024.pdf
 - C.A.316_2022.pdf
 - C.P.L.A.552-K_2021.pdf
 - C.P.L.A.3210-L_2023.pdf
 - C.P.L.A.607_2021.pdf
 - C.P.L.A.520-K_2024.pdf
 - C.P.L.A.4649_2022.pdf
 - C.P.L.A.3209_2020.pdf
 - C.P.L.A.824-K_2023.pdf
 - C.P.L.A.463_2024.pdf
 - C.P.L.A.2007-L_2023.pdf
 - C.P.L.A.3755_2022.pdf
 - Crl.P.L.A.220_2024.pdf
 - C.A.730_2015.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only 

In [13]:
query = "What was the dispute in C.A.1509 of 2021?"
answer = agent.search(query, k=20)



🔍 QUERY: What was the dispute in C.A.1509 of 2021?
📋 Detected case number: C.A.1509_2021
✅ Found exact case match

📄 Retrieved PDFs:
 - C.A.1509_2021.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)


BEGIN NOW.
            
            

RETRIEVED DOCUMENTS:
From C.A.1509_2021.pdf:
[Case Type: C.A.1509] [Year: 2021] [Case Number: C.A.1509_2021]

I.C.A.No.34/2017, W.P.No.2801/2015, I.C.A.Nos.82/2020, 140/2020, 98/2020, 
103/2020, 1 04/2020, 118/2020, 119/2020, 120/2020, 280/2021, 286/2021, 
84/2020, 87/2020, 88/2020, 89/2020, 91/2020, 92/2020, 95/2020, 109/2020, 
149/2020, 294/2021 , 117/2020, 150/2020, 283/2021, 274/2020 , 27

In [15]:
query = "What were the problems discussed in Civil Appeal No 1509 of 2021?"
answer = agent.search(query, k=20)



🔍 QUERY: What were the problems discussed in Civil Appeal No 1509 of 2021?

📄 Retrieved PDFs:
 - C.A.1509_2021.pdf
 - C.A.119-L_2022.pdf
 - C.A.152_2019.pdf
 - C.P.L.A.2009_2025.pdf
 - C.A.117-K_2022.pdf
 - C.A.730_2015.pdf
 - C.A.1980_2023.pdf
 - C.A.455_2012.pdf
 - C.A.982_2018.pdf
 - C.A.26_2015.pdf
 - C.A.350_2016.pdf
 - C.A.634_2018.pdf
 - C.A.1088_2013.pdf
 - Crl.P.L.A.1315-L_2022.pdf
 - C.P.2_2022.pdf
 - C.A.239-L_2018.pdf
 - Crl.P.L.A.1124-L_2015.pdf
 - C.P.L.A.1189_2025.pdf
 - C.A.1507_2024.pdf
 - C.A.316_2022.pdf


KeyboardInterrupt: 

In [16]:
query = "What were the findings of the High Court were challenged before the Supreme Court in CPLA No. 210 of 2024?"
answer = agent.search(query, k=20)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🔍 QUERY: What were the findings of the High Court were challenged before the Supreme Court in CPLA No. 210 of 2024?

📄 Retrieved PDFs:
 - C.P.L.A.1106_2024.pdf
 - C.P.L.A.251-Q_2024.pdf
 - C.P.L.A.1189_2025.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.2009_2025.pdf
 - I.C.A.1_2024.pdf
 - C.P.L.A.552-K_2021.pdf
 - C.P.L.A.607_2021.pdf
 - C.P.L.A.463_2024.pdf
 - C.P.L.A.3210-L_2023.pdf
 - C.A.316_2022.pdf
 - C.P.L.A.4649_2022.pdf
 - C.P.L.A.520-K_2024.pdf
 - C.P.L.A.824-K_2023.pdf
 - C.P.L.A.210_2024.pdf
 - C.P.L.A.2007-L_2023.pdf
 - C.P.L.A.3755_2022.pdf
 - C.P.L.A.3209_2020.pdf
 - C.A.730_2015.pdf


KeyboardInterrupt: 

In [18]:
query = "Why did the Supreme Court dismiss the CPLA No. 210 of year 2024?"
answer = agent.search(query, k=20)


🔍 QUERY: Why did the Supreme Court dismiss the CPLA No. 210 of year 2024?

📄 Retrieved PDFs:
 - C.P.L.A.3755_2022.pdf
 - C.P.L.A.3209_2020.pdf
 - C.R.P.1077_2023.pdf
 - C.P.L.A.251-Q_2024.pdf
 - C.P.L.A.2007-L_2023.pdf
 - C.A.316_2022.pdf
 - C.A.1509_2021.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.694-P_2024.pdf
 - C.P.L.A.78-K_2024.pdf
 - C.P.L.A.520-K_2024.pdf
 - C.P.L.A.148-L_2024.pdf
 - C.P.L.A.121_2024.pdf
 - C.P.L.A.2712_2020.pdf
 - C.P.L.A.277-Q_2024.pdf
 - C.P.L.A.5029_2024.pdf
 - C.P.L.A.2477_2024.pdf
 - C.P.L.A.1573_2024.pdf
 - C.P.L.A.6-L_2023.pdf


KeyboardInterrupt: 

In [21]:
query = "Why did the Supreme Court dismiss the Civil Petition to Leave Appeal No. 210 in the year 2024?"
answer = agent.search(query, k=25)


🔍 QUERY: Why did the Supreme Court dismiss the Civil Petition to Leave Appeal No. 210 in the year 2024?

📄 Retrieved PDFs:
 - C.P.L.A.5029_2024.pdf
 - C.P.L.A.862_2024.pdf
 - C.P.L.A.277-Q_2024.pdf
 - C.P.L.A.3920_2024.pdf
 - C.P.L.A.552-K_2021.pdf
 - C.P.L.A.1737-L_2020.pdf
 - C.P.L.A.1057_2019.pdf
 - C.P.L.A.3824_2023.pdf
 - C.P.L.A.4194_2023.pdf
 - C.P.L.A.148-L_2024.pdf
 - C.P.L.A.2477_2024.pdf
 - C.P.L.A.3649_2023.pdf
 - C.R.P.557_2020.pdf
 - C.P.L.A.662-K_2024.pdf
 - C.P.L.A.1970-L_2024.pdf
 - C.P.L.A.3578_2024.pdf
 - C.P.L.A.312_2025.pdf
 - C.P.L.A.5718_2021.pdf
 - C.P.L.A.1189_2025.pdf
 - C.P.L.A.1573_2024.pdf
 - C.P.L.A.181-Q_2021.pdf
 - C.P.L.A.2477-L_2015.pdf
 - C.P.L.A.516-K_2022.pdf
 - C.R.P.540_2023.pdf
 - Crl.A.144-L_2020.pdf


KeyboardInterrupt: 

In [22]:
query = "I need cases related to taxation, handled by Justice Yahya Afridi"
answer = agent.search(query, k=25)


🔍 QUERY: I need cases related to taxation, handled by Justice Yahya Afridi

📄 Retrieved PDFs:
 - C.R.P.296_2020.pdf
 - C.A.1089_2015.pdf
 - C.A.243_2011.pdf
 - C.A.1422_2019.pdf
 - C.P.L.A.3472_2023.pdf
 - C.R.P.275_2022.pdf
 - C.A.1660_2014.pdf
 - C.P.L.A.78-K_2024.pdf
 - C.A.630_2010.pdf
 - C.P.L.A.862_2024.pdf
 - C.A.1262_2018.pdf
 - C.P.L.A.3578_2024.pdf
 - C.A.350_2016.pdf
 - C.P.L.A.4177_2024.pdf
 - J.P.516_2018.pdf
 - C.P.L.A.339-L_2023.pdf
 - C.A.649_2019.pdf
 - C.A.247_2021.pdf
 - C.A.458_2017.pdf
 - C.A.1032_2018.pdf
 - C.A.25-Q_2018.pdf
 - C.P.24_2023.pdf
 - C.A.1191_2014.pdf
 - C.P.L.A.3155-L_2023.pdf
 - C.P.L.A.47_2024.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many 

In [23]:
query = "I need cases related to land acquisition, handled by Justice Yahya Afridi"
answer = agent.search(query, k=5)


🔍 QUERY: I need cases related to land acquisition, handled by Justice Yahya Afridi

📄 Retrieved PDFs:
 - C.A.1980_2023.pdf
 - C.A.538_2022.pdf
 - C.A.914-L_2013.pdf
 - C.A.2186_2017.pdf
 - C.A.25-Q_2018.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)


BEGIN NOW.
            
            

RETRIEVED DOCUMENTS:
From C.A.1980_2023.pdf:
[Case Type: C.A.1980] [Year: 2023] [Case Number: C.A.1980_2023]

Mr. Qazi Ghulam Rauf, ASC 
Mr. Junaid Ammar, ASC  
 
  
   
Date of Hearing : 08.04.2024  
 
JUDGMENT 
 
  YAHYA AFRIDI, J. - The Government of Pakistan, through 
the Secretary of the Ministry of Defence (“appellant”), has filed the